# Bloodflow: does aggregation turn Poiseuille flow into plug flow?

A toy model: rigid disks in a 2D channel under Poiseuille flow. Turn on a
short-range (Morse) attraction and the disks aggregate into clusters that
move as a plug, carrying slow near-wall particles along — cutting the long
tail of transit times.

This notebook runs entirely in your browser (JupyterLite + Pyodide).
The two runs below take about a minute.

In [ ]:
import numpy as np
from dataclasses import replace
from bloodflow import Params, cluster_stats, simulate, transit_summary

base = Params(
    N=60, a=1.0, H=16.0, L=40.0, eta=1.0, Umax=1.0,
    kappa=2.0, eps_rep=1.0, kT=0.0, dt=0.02, seed=1,
)
results = {}
for eps in (0.0, 2.0):
    p = replace(base, eps=eps)
    res = simulate(p, n_steps=8000, equil_steps=500)
    s = transit_summary(res.transit, res.wall_time)
    c = cluster_stats(res.final_pos, cutoff=2.0 * p.a + 0.5 / p.kappa)
    results[eps] = (res, s, c)
    print(f"eps={eps}: mean={s['mean']:.1f}  median={s['median']:.1f}  "
          f"p95={s['p95']:.1f}  wall residence={s['mean_wall_residence']:.1f}  "
          f"mean cluster={c['mean_cluster_size']:.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for eps, color in ((0.0, "tab:blue"), (2.0, "tab:orange")):
    res, s, c = results[eps]
    done = res.transit[~np.isnan(res.transit)]
    ax.hist(done, bins=25, alpha=0.6, color=color,
            label=f"eps={eps} (p95={s['p95']:.0f})")
ax.set_xlabel("transit time")
ax.set_ylabel("particles")
ax.legend()
ax.set_title("Transit-time distribution: no attraction vs aggregation")
plt.show()

## Takeaway

With attraction on (eps = 2), the long tail of slow transits shrinks
(p95 138 → 75), near-wall residence collapses (35 → 4), and clusters grow
(mean size 1.5 → 15). The aggregates move as a plug.

See also: the [pulsatile-flow video](videos/bloodflow-pulsatile.mp4)
(heartbeat version — the plug breathes but survives) and the
[full write-up](https://github.com/marcocaggioni/bloodflow/blob/main/README.md).

## Export an MP4 in the browser

JupyterLite ships no ffmpeg binary, so a notebook cannot normally write a
video file. This section works around that with
[ffmpeg.wasm](https://ffmpegwasm.netlify.app/) — FFmpeg compiled to
WebAssembly, loaded from a CDN on first use (~25 MB, cached by the browser
afterwards) and driven from Python through Pyodide's JavaScript bridge.
The single-threaded core is used on purpose: GitHub Pages sends no
cross-origin-isolation headers, so the multi-threaded core cannot load here.

Run the two cells below: the first renders the eps = 2 run as 90 PNG
frames (3 s at 30 fps), the second encodes them with libx264 and saves
`bloodflow-export.mp4` next to this notebook — it appears in the file
browser (right-click → Download) and plays inline below. The temporary
PNG frames are deleted from ffmpeg's in-memory filesystem after encoding;
the simulation data stays in the notebook.

In [ ]:
# Render the eps = 2 run as PNG frames (kept in memory) for the MP4 export below.
# Same seed / equilibration as the sweep above, so this is that run's plug forming,
# re-run short (500 equil + 1800 steps) just for the animation.
import io

import numpy as np
import matplotlib.pyplot as plt
from bloodflow import Params, core

FPS = 30
N_FRAMES = 90
STRIDE = 20            # steps between frames (0.4 time units per frame)
EQUIL_STEPS = 500

p_anim = Params(N=60, a=1.0, H=16.0, L=40.0, eta=1.0, Umax=1.0,
                kappa=2.0, eps_rep=1.0, kT=0.0, dt=0.02, seed=1, eps=2.0)
rng = np.random.default_rng(p_anim.seed)
pos = core.init_positions(p_anim, rng)
for _ in range(EQUIL_STEPS):
    core._step(pos, p_anim, rng, 0.0)

fig, ax = plt.subplots(figsize=(10, 4), dpi=80)
ax.set_xlim(0, p_anim.L)
ax.set_ylim(0, p_anim.H)
ax.set_aspect("equal")
ax.set_xlabel("x (flow direction)")
ax.set_ylabel("y (channel width)")
ax.set_title("eps = 2 (aggregation) — plug flow")
fig.canvas.draw()
# Marker area (points^2) matching a disk of radius a in data units.
p0 = ax.transData.transform((0.0, 0.0))
p1 = ax.transData.transform((2.0 * p_anim.a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
scat = ax.scatter(pos[:, 0], pos[:, 1], s=size, c="tab:orange", edgecolors="none")
time_text = ax.text(0.01, 0.95, "", transform=ax.transAxes, va="top", fontsize=10)

frames_png = []
for frame in range(N_FRAMES):
    if frame > 0:
        for _ in range(STRIDE):
            core._step(pos, p_anim, rng, 0.0)
    scat.set_offsets(pos)
    time_text.set_text(f"t = {(EQUIL_STEPS + frame * STRIDE) * p_anim.dt:.1f}")
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_png.append(buf.getvalue())
plt.close(fig)
print(f"rendered {len(frames_png)} frames "
      f"({len(frames_png) / FPS:.1f} s at {FPS} fps, eps = 2.0)")

In [ ]:
# Encode the frames above into an MP4, entirely in the browser.
# JupyterLite has no ffmpeg binary, so this cell loads ffmpeg.wasm
# (FFmpeg compiled to WebAssembly) from a CDN and drives it from Python
# through Pyodide's JavaScript bridge. First run downloads ~25 MB;
# the browser caches it afterwards. Single-threaded core on purpose:
# GitHub Pages sends no cross-origin-isolation headers, so the
# multi-threaded core cannot load here.
from pyodide.code import run_js

try:
    n_frames = len(frames_png)
    fps = FPS
except NameError:
    raise RuntimeError(
        "Run the 'Render frames' cell above first — it defines frames_png and FPS."
    )

FFMPEG_VERSION = "0.12.15"   # @ffmpeg/ffmpeg; expects @ffmpeg/core 0.12.10
UTIL_VERSION = "0.12.2"      # @ffmpeg/util (toBlobURL)
CORE_VERSION = "0.12.10"     # @ffmpeg/core, single-threaded UMD build
OUT_NAME = "bloodflow-export.mp4"

ffmpeg_mod = await run_js(f"import('https://esm.sh/@ffmpeg/ffmpeg@{FFMPEG_VERSION}')")
util_mod = await run_js(f"import('https://esm.sh/@ffmpeg/util@{UTIL_VERSION}')")
ffmpeg = ffmpeg_mod.FFmpeg.new()
core_base = f"https://unpkg.com/@ffmpeg/core@{CORE_VERSION}/dist/umd"
# Never pass CDN URLs to load() directly: browsers refuse to build a Worker
# from a cross-origin URL and load() would hang forever. toBlobURL fetches
# the files and wraps them in same-origin blob URLs.
core_url = await util_mod.toBlobURL(f"{core_base}/ffmpeg-core.js", "text/javascript")
wasm_url = await util_mod.toBlobURL(f"{core_base}/ffmpeg-core.wasm", "application/wasm")
print("loading ffmpeg.wasm core (first run downloads ~25 MB)...")
await ffmpeg.load({"coreURL": core_url, "wasmURL": wasm_url})
print("ffmpeg.wasm loaded")

for i, png in enumerate(frames_png):
    await ffmpeg.writeFile(f"frame_{i:04d}.png", png)
frames_png.clear()  # free the Python-side copies; the frames now live in ffmpeg's memory FS
print(f"wrote {n_frames} frames to ffmpeg's in-memory filesystem")

rc = await ffmpeg.exec([
    "-y", "-framerate", str(fps), "-i", "frame_%04d.png",
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20",
    "-movflags", "+faststart", OUT_NAME,
])
if rc != 0:
    raise RuntimeError(f"ffmpeg exited with code {rc}")
mp4_bytes = bytes((await ffmpeg.readFile(OUT_NAME)).to_py())
with open(OUT_NAME, "wb") as f:
    f.write(mp4_bytes)
print(f"wrote {OUT_NAME} ({len(mp4_bytes)} bytes) — "
      "find it in the file browser (right-click → Download)")

# Cleanup: delete the temporary frames (and the encoded copy) from
# ffmpeg's in-memory filesystem. The simulation data (results) and the
# saved MP4 are untouched.
for i in range(n_frames):
    await ffmpeg.deleteFile(f"frame_{i:04d}.png")
await ffmpeg.deleteFile(OUT_NAME)
print(f"deleted {n_frames} temporary frames from ffmpeg's filesystem")

from IPython.display import Video, display
try:
    display(Video(OUT_NAME, embed=True))
except Exception as exc:  # inline preview is a bonus; the file is already saved
    print(f"(inline preview unavailable: {exc} — download {OUT_NAME} from the file browser)")